In [1]:
# Import necessary libraries
import os
import sys
import shutil
import glob

import numpy as np

from tqdm.contrib.concurrent import thread_map# or thread_map

sys.path.append("/home/klymy/projects/researches/data-exploration/video_splitting")
from splitter import create_project_file

In [2]:
def get_files_glob(directory, ext="*.txt"):
    pattern = os.path.join(directory, '**', ext)
    txt_files = glob.glob(pattern, recursive=True)
    return txt_files

In [3]:
MARGIN = 0.2

BASE_PTH = "/home/klymy/data/people_detection"
BASE_DATASET = "filtered_v2_10_sec"
FILTERED_DATASET = f"filtered_v3_margin_{int(MARGIN * 100)}"

SUBSETS = ["train", "val", "holdout", "algotest"]

In [4]:
src_pth = os.path.join(BASE_PTH, BASE_DATASET)
dst_pth = os.path.join(BASE_PTH, FILTERED_DATASET)
if os.path.exists(dst_pth):
    shutil.rmtree(dst_pth)
shutil.copytree(src_pth, dst_pth)

cache_files = get_files_glob(dst_pth, "*.cache")
for i in cache_files:
    os.remove(i)

In [5]:
# create project file
create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"}, [1, 2])
create_project_file(
    os.path.join(dst_pth, "project_holdout.yaml"), dst_pth, {0: "person"}, [1, 2], "images/holdout")
create_project_file(
    os.path.join(dst_pth, "project_algotest.yaml"), dst_pth, {0: "person"}, [1, 2], "images/algotest")

In [6]:
labels_pths = get_files_glob(os.path.join(dst_pth, "labels"))
len(labels_pths)

119833

In [7]:
def get_min_margin(x, y, widths, heights):
    # Calculate the margin sizes
    left_margin = x + widths / 2  # Distance from right side of the box to the left edge of the frame
    right_margin = 1 - (x - widths / 2)  # Distance from left side of the box to the right edge of the frame
    top_margin = y + heights / 2  # Distance from bottom side of the box to the top edge of the frame
    bottom_margin = 1 - (y - heights / 2)  # Distance from top side of the box to the bottom edge of the frame
    
    return np.min([left_margin, right_margin, top_margin, bottom_margin])

def process_file(pth):
    filtered_lines = []

    count = 0
    count_filtered = 0
    with open(pth, "rt") as f:
        for line in f.readlines():
            count +=1
            _, x_c, y_c, w, h, _, _ = [float(i) for i in line.split()]
            
            min_margin = get_min_margin(x_c, y_c, w, h)
            if min_margin > MARGIN:
                filtered_lines.append(line)
                count_filtered += 1

    with open(pth, "wt") as f:
        f.writelines(filtered_lines)

    return count, count_filtered

In [8]:
r = thread_map(process_file, labels_pths, max_workers=8)

count = 0
count_filtered = 0
for i, j in r:
    count += i
    count_filtered += j

print(f"Margin {MARGIN} removes {(count - count_filtered) / count * 100:.02f}% bboxes")

  0%|          | 0/119833 [00:00<?, ?it/s]

Margin 0.2 removes 21.55% bboxes


In [9]:
#  Margin 0.1 removes 6% bboxes
#  Margin 0.15 removes 12% bboxes
#  Margin 0.2 removes 21.55% bboxes